# 03 · Advanced Analytics — Statistical & Modelling Deep-Dives

Notebooks `00`–`02` **built and exported** the warehouse. This notebook is where the
**analysis** happens: statistical and machine-learning methods that go beyond what a
Power BI measure can express. Each section is a self-contained investigation — a business
**question**, the **method**, the **code**, and a written **finding**.

| # | Method | Question it answers | Where the result lives |
|---|---|---|---|
| **3.1** | Gini Coefficient | How concentrated is customer spending? | Finding (this notebook) |
| **3.2** | Hypothesis Testing | Are two segments *really* different, or is it noise? | Finding (this notebook) |
| **3.3** | RFM Segmentation (K-means) | What natural customer segments exist? | → back into Power BI |
| **3.4** | Market Basket Analysis | Which products sell together? | Finding (this notebook) |
| **3.5** | Return Propensity (logistic regression) | Which orders are likely to be returned? | → back into Power BI |
| **3.6** | A/B-Test Power Plan | How large a test do we need to detect an effect? | Finding (this notebook) |
| **3.7** | *(optional)* CLV / Survival Analysis | What is a customer worth over their lifetime? | Finding (this notebook) |

**Convention:** every section opens its own **read-only** DuckDB connection, pulls only
the columns it needs, and closes the connection at the end — so sections run
independently and never lock the warehouse against the pipeline notebook.

---

## 3.1 Gini Coefficient

**Question:** How concentrated is customer spending overall — do a small share of
customers drive a disproportionate share of revenue?

### Step 1 - Connect and pull the data

Read-only connection so it never conflicts with the pipeline notebook.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

connection = duckdb.connect("../artifacts/practice_analytics.duckdb", read_only=True)

customers = connection.execute(
    "SELECT customer_id, net_sales_value, country, acquisition_source FROM mart.customer_360"
).fetchdf()

print(f"Loaded {len(customers):,} customers")
customers.head()

### Step 2 - Write the Gini function, and validate it on a known example first

Before trusting it on real data, test it on cases where you already know the answer -
same "check against a known answer" habit used throughout the SQL/DAX work.

In [ ]:
def gini(values):
    values = np.sort(np.asarray(values, dtype=float))
    n = len(values)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * values) - (n + 1) * np.sum(values)) / (n * np.sum(values))

# Sanity checks before trusting it on real data
assert round(gini([10, 10, 10, 10]), 4) == 0.0          # perfect equality -> 0
print("Perfect equality check:", gini([10, 10, 10, 10]))

print("Extreme inequality check (should be close to 1):", gini([0, 0, 0, 100]))
print("Mild inequality check (should be a small positive number):", gini([8, 9, 10, 13]))

### Step 3 - Compute the real, overall Gini coefficient

This is the headline number for the whole customer base.

In [ ]:
overall_gini = gini(customers["net_sales_value"])
print(f"Overall customer spending Gini coefficient: {overall_gini:.3f}")

### Step 4 - Explore across a few interesting cuts

Computing Gini per segment reveals *where* concentration is strongest, not just
that it exists overall.

In [ ]:
for col in ["country", "acquisition_source"]:
    print(f"\nGini by {col}:")
    segment_gini = (
        customers.groupby(col)["net_sales_value"]
        .apply(gini)
        .sort_values(ascending=False)
    )
    print(segment_gini)

### Step 5 - Plot the Lorenz curve

The formal picture behind the Gini number: cumulative % of customers (x-axis) vs.
cumulative % of revenue (y-axis). The further the curve bows below the 45-degree
"perfect equality" line, the more concentrated spending is.

In [ ]:
sorted_vals = np.sort(customers["net_sales_value"].to_numpy())
cum_customers = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
cum_revenue = np.cumsum(sorted_vals) / sorted_vals.sum()

plt.figure(figsize=(6, 6))
plt.plot(cum_customers, cum_revenue, label="Lorenz curve", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect equality")
plt.xlabel("Cumulative % of customers")
plt.ylabel("Cumulative % of revenue")
plt.title(f"Lorenz Curve (Gini = {overall_gini:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

### Step 6 - Write the findings

Fill this in after running the cells above. Cover:

- **The overall Gini number** and what it means in plain English (closer to 1 = more
  concentrated spending among fewer customers).
- **Which segment(s) stood out** in the Step 4 breakdown - did any country or
  acquisition source show noticeably higher/lower concentration?
- **One caveat** - this is synthetic data, so the concentration pattern reflects
  the dataset's generation process, not necessarily a real market's dynamics.

> *(Your written conclusion goes here once you've seen the actual numbers.)*

### Step 7 - Close the connection

In [ ]:
connection.close()
print("Connection closed.")

---

## 3.2 Hypothesis Testing

Section 3.1 measured *how concentrated* spending is. This section asks a different kind of
question: when we split customers into groups and the groups *look* different, is the
difference **real**, or just random sampling noise? That is what hypothesis testing decides.

"Comparing groups" comes in three shapes, each with its own correct test — so this section
is really three mini-analyses that together form the standard group-comparison toolkit:

| # | Test | Compares | Question here |
|---|---|---|---|
| **3.2.1** | Two-sample **t-test** | 2 group means | Do **male vs female** customers differ in AOV? |
| **3.2.2** | **ANOVA** | 3+ group means | Does AOV differ across the **5 acquisition channels**? |
| **3.2.3** | **Chi-square** | categorical proportions | Does **return rate** differ across product **categories**? |

Every test follows the same ritual: state H0/H1 -> pick alpha -> compute a **p-value**
(*is the difference real?*) -> compute an **effect size** (*is it big enough to matter?*).
Watch how, on this synthetic data, the answer is almost always "**detectable at most, but
negligible in size**" — the single most important lesson in not chasing significance alone.

### 3.2.1 Two-sample t-test — male vs female AOV

**Question:** Do **male** and **female** customers spend a different amount **per order**
(AOV)? The two-sample t-test compares the *means* of a continuous number between two
independent groups — the natural fit for a dimension with exactly two values.

### Step 1 - State the hypotheses (before touching the data)

A hypothesis test always starts by writing down two competing claims **before** you look
at the result, so you can't be tempted to move the goalposts afterwards.

- **Null hypothesis (H0):** male and female customers have the **same** mean AOV.
  Any difference we see in the sample is just random noise. *(This is the "nothing
  interesting is happening" default — assumed true until the data forces us to reject it.)*
- **Alternative hypothesis (H1):** the two groups have a **different** mean AOV.

**Significance level (alpha) = 0.05** — the risk we accept of wrongly rejecting H0
(a "false alarm": claiming a difference that isn't real). 0.05 = "I'll accept a 5% chance."

**Two-sided test:** we ask "is it *different*" (could be higher or lower), not "is it
*higher*", so we test both directions.

The test produces a **p-value**: the probability of seeing a gap this big (or bigger)
*if H0 were true*. A small p-value means the data would be very surprising under
"no difference", so we reject H0. Decision rule: **p < alpha -> reject H0** (the
difference is statistically significant).

### Step 2 - Connect and pull the two groups

Its own read-only connection. We pull just two columns from `customer_360`: the outcome
we're testing (`average_order_value`) and the column that splits customers into the two
groups (`gender`). We keep only `'M'` and `'F'` rows so the two groups are clean.

In [ ]:
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

connection = duckdb.connect("../artifacts/practice_analytics.duckdb", read_only=True)

df = connection.execute(
    "SELECT average_order_value, gender FROM mart.customer_360 WHERE gender IN ('M', 'F')"
).fetchdf()

# Split into the two independent groups
male   = df.loc[df["gender"] == "M", "average_order_value"].to_numpy()
female = df.loc[df["gender"] == "F", "average_order_value"].to_numpy()

print(f"Male customers   : {len(male):,}")
print(f"Female customers : {len(female):,}")

### Step 3 - Look before you test (describe and visualise)

Never jump straight to the test. First look at the two groups' means, spreads, and shapes
— the test only *formalises* what your eyes should already suspect, and a plot catches
problems (extreme skew, weird outliers) that a single p-value would hide.

In [ ]:
for name, group in [("Male", male), ("Female", female)]:
    print(f"{name:7s}  n={len(group):>6,}  mean=${group.mean():7.2f}  "
          f"median=${np.median(group):7.2f}  std=${group.std(ddof=1):7.2f}")

observed_gap = male.mean() - female.mean()
print(f"\nObserved difference in mean AOV: ${observed_gap:,.2f}")

# Density histograms (density=True) so the two groups are comparable despite different sizes.
# Cap the x-axis at the 99th percentile so a few extreme values don't squash the picture.
plt.figure(figsize=(8, 4))
bins = np.linspace(0, np.percentile(df["average_order_value"], 99), 40)
plt.hist(female, bins=bins, alpha=0.6, label="Female", density=True)
plt.hist(male,   bins=bins, alpha=0.6, label="Male",   density=True)
plt.xlabel("Average order value ($)")
plt.ylabel("Density")
plt.title("AOV distribution: male vs female customers")
plt.legend()
plt.tight_layout()
plt.show()

### Step 4 - Run the two-sample t-test (Welch's)

`scipy.stats.ttest_ind` compares two group means. We pass `equal_var=False`, which runs
**Welch's t-test** — the safer default that does **not** assume the two groups have equal
variance (real-world groups rarely do). It returns:

- **t-statistic** — how many standard errors apart the two means are. Bigger magnitude =
  groups further apart relative to the noise.
- **p-value** — the probability of a gap this large if H0 (no real difference) were true.

In [ ]:
t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)

print(f"t-statistic: {t_stat:.3f}")
print(f"p-value    : {p_value:.3g}")

alpha = 0.05
if p_value < alpha:
    print(f"\np < {alpha}  ->  REJECT H0: the difference in mean AOV is statistically significant.")
else:
    print(f"\np >= {alpha}  ->  FAIL TO REJECT H0: no statistically significant difference.")

### Step 5 - Measure the effect size (is the difference big enough to matter?)

Here's the catch every analyst must understand: with a large sample (we have ~80,000
customers), the t-test will call **even a tiny, business-irrelevant difference**
"statistically significant". A p-value tells you the difference is *real*; it does **not**
tell you it's *big*.

So we also compute **Cohen's d** — the size of the gap measured in standard deviations,
independent of sample size. Rough convention:

| Cohen's d | Practical meaning |
|---|---|
| ~0.2 | small |
| ~0.5 | medium |
| ~0.8 | large |

A significant p-value **and** a meaningful d = a difference worth acting on. Significant p
but d ~ 0.05 = "real, but too small to care about".

In [ ]:
# Cohen's d using a pooled standard deviation
n1, n2 = len(male), len(female)
pooled_std = np.sqrt(
    ((n1 - 1) * male.var(ddof=1) + (n2 - 1) * female.var(ddof=1)) / (n1 + n2 - 2)
)
cohens_d = (male.mean() - female.mean()) / pooled_std

print(f"Cohen's d: {cohens_d:.3f}")

magnitude = (
    "negligible" if abs(cohens_d) < 0.2 else
    "small"      if abs(cohens_d) < 0.5 else
    "medium"     if abs(cohens_d) < 0.8 else
    "large"
)
print(f"Effect size is {magnitude}.")

### Step 6 - Write the findings

Fill this in after running the cells above. A complete finding states **all four**:

- **The decision** — did we reject H0? (p vs alpha)
- **The direction & size** — which group is higher, by how many dollars, and how big is
  Cohen's d?
- **The business reading** — does the difference actually matter for a decision, or is it
  "significant but trivial"? With ~80k customers, watch for a tiny p-value paired with a
  negligible Cohen's d.
- **One caveat** — this is synthetic data, so any gender gap reflects the generator's
  design, not a real behavioural difference; never build a real pricing/targeting policy
  on a single test like this.

> *(Your written conclusion goes here once you've seen the actual numbers.)*

### Step 7 - Close the connection

In [ ]:
connection.close()
print("Connection closed.")

### 3.2.2 ANOVA — AOV across the 5 acquisition channels

A t-test only compares **two** groups. With **five** channels (Display, Email, Facebook,
Organic, Search) we'd need 10 separate pairwise t-tests to check every combination — and
each test carries a 5% false-alarm risk, so running many of them almost guarantees a fluke
"significant" result (the **multiple-comparisons** problem). **ANOVA** (Analysis of Variance)
solves this with a single test for *"do **any** of the group means differ?"*

- **H0:** all five channels have the **same** mean AOV.
- **H1:** **at least one** channel's mean AOV differs.
- **alpha = 0.05.** The statistic is **F** = (variation *between* groups) / (variation
  *within* groups). A large F with a small p means the groups sit further apart than the
  within-group noise can explain.

In [ ]:
import duckdb
import numpy as np
from scipy import stats

connection = duckdb.connect("../artifacts/practice_analytics.duckdb", read_only=True)
chan = connection.execute(
    "SELECT average_order_value, acquisition_source FROM mart.customer_360"
).fetchdf()
connection.close()

# One AOV array per channel — the input ANOVA expects
groups = [g["average_order_value"].to_numpy() for _, g in chan.groupby("acquisition_source")]
labels = [name for name, _ in chan.groupby("acquisition_source")]

for name, g in zip(labels, groups):
    print(f"{name:9s}  n={len(g):>6,}  mean AOV=${g.mean():.2f}")

f_stat, p_value = stats.f_oneway(*groups)
print(f"\nF-statistic: {f_stat:.3f}")
print(f"p-value    : {p_value:.3g}")
print("REJECT H0" if p_value < 0.05 else "FAIL TO REJECT H0  (channels look interchangeable)")

**Effect size — eta-squared (eta^2).** ANOVA's answer to Cohen's d: the share of the total
variation in AOV that is explained by *which channel* a customer came from. Rough guide:
~0.01 small, ~0.06 medium, ~0.14 large. If eta^2 is near 0, channel tells you essentially
nothing about how much a customer spends per order.

In [ ]:
grand_mean = chan["average_order_value"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = ((chan["average_order_value"] - grand_mean) ** 2).sum()
eta_sq = ss_between / ss_total
print(f"eta-squared: {eta_sq:.4f}")

**Finding (3.2.2):** *(fill after running)* F ≈ ____, p ≈ ____, eta^2 ≈ ____. Do the five
channels differ in AOV — and even if p is small, is eta^2 large enough to matter?

### 3.2.3 Chi-square — return rate across product categories

The first two tests compared a **numeric average** (AOV). Return behaviour is different:
each item is either **returned or not** — a yes/no **proportion**, not an average. For
*"is a categorical outcome (returned?) related to a categorical group (product category)?"*
the right tool is the **chi-square test of independence**, run on a table of **counts**.

- **H0:** return rate is the **same** across all categories — returning is *independent* of
  category.
- **H1:** **at least one** category has a different return rate.
- We build a **contingency table** (category × returned / not-returned) from return-eligible
  items, and `chi2_contingency` compares the **observed** counts to the counts H0 predicts.

In [ ]:
import duckdb
import numpy as np
from scipy import stats

connection = duckdb.connect("../artifacts/practice_analytics.duckdb", read_only=True)
ct = connection.execute(
    """
    SELECT p.category,
           SUM(f.returned_item_flag)                                 AS returned,
           SUM(CASE WHEN f.returned_item_flag = 0 THEN 1 ELSE 0 END) AS not_returned
    FROM core.fact_order_item f
    JOIN core.dim_product p ON f.product_id = p.product_id
    WHERE f.return_observation_eligible_flag = 1
    GROUP BY p.category
    HAVING COUNT(*) > 200
    """
).fetchdf()
connection.close()

ct["return_rate"] = ct["returned"] / (ct["returned"] + ct["not_returned"])
print("Highest / lowest return-rate categories:")
print(ct.sort_values("return_rate", ascending=False).head(3).to_string(index=False))
print(ct.sort_values("return_rate").head(3).to_string(index=False))

# The contingency table is just the two count columns
table = ct[["returned", "not_returned"]].to_numpy()
chi2, p_value, dof, expected = stats.chi2_contingency(table)
print(f"\nchi2 = {chi2:.2f}   dof = {dof}   p-value = {p_value:.3g}")
print("REJECT H0" if p_value < 0.05 else "FAIL TO REJECT H0  (return rate is flat across categories)")

**Effect size — Cramer's V.** The chi-square counterpart to Cohen's d: the strength of the
association between category and returning, on a 0–1 scale (~0.1 small, ~0.3 medium, ~0.5
large). Near 0 means category and return behaviour are essentially unrelated — a significant
p with V ≈ 0 is again "real but trivial".

In [ ]:
n = table.sum()
k = min(table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))
print(f"Cramer's V: {cramers_v:.4f}")

**Finding (3.2.3):** *(fill after running)* chi2 ≈ ____, p ≈ ____, Cramer's V ≈ ____.

### 3.2 Findings — synthesis

*(Fill in after running all three tests.)* Across three different lenses — gender (t-test),
channel (ANOVA), category (chi-square) — the pattern is consistent:

- **t-test (gender AOV):** the gap is *statistically significant* (p is tiny) but the
  **effect size is negligible** (Cohen's d ≈ ____) — a textbook case of a large sample making
  a trivial gap "significant".
- **ANOVA (channel AOV):** the channels are **not even statistically different** (p ≈ ____,
  eta^2 ≈ ____).
- **Chi-square (category return rate):** return rate is **flat across categories**
  (p ≈ ____, Cramer's V ≈ ____).

**Business reading:** on these dimensions the customer base is statistically **homogeneous**
— which echoes the Gini result (concentration is uniform across segments). The actionable
lever is *not* demographic / channel / category targeting, but the **within-customer
concentration** from 3.1: focus on the high-value tail, not on "men vs women" or
"Email vs Facebook".

**Caveat:** this is synthetic data, so the flatness reflects theLook's generator, not a real
market — in real data you would expect at least some of these tests to find a genuine,
sizeable difference. The *method* (test -> p-value -> effect size -> honest read) is what
transfers.